# 07. 2025년 test (test_2025)

**이 노트북이 하는 일**

모든 설정이 확정된 뒤 **2025년 test를 이번 한 번만** 평가합니다. 설정은 `00_settings`에서 **그대로 고정**합니다 (차수 (1,0,1)(1,1,1,7), `MASK_COLS`, 계단 더미, `summer_vacation`, 충격 k=`SHOCK_K`=1, 회복 m=`RECOVERY_M`=2). **2025년 결과를 보고 설정을 바꾸지 않습니다.**

> **실행 기록**: 이 노트북은 2025년 test를 **한 번만** 실행했습니다. 실행 일시: **2026-11-13 16:34:19+09:00** (`results/test2025_run_info.json`에도 기록). 실행하면 이 기록 파일이 만들어지고, 파일이 있으면 같은 노트북을 다시 실행하지 못하게 막습니다.

**하는 일**

1. **재학습**: Naive(팀원 노트북 차수), N, A, B_lag, B, C를 **2014~2024로 다시 학습**해 계수를 고정합니다. 학습 기간에 값이 전부 0인 더미는 뺍니다 (`fare_2025`).
2. **2025년 예측 평가**: (가) 하루 앞 롤링과 (나) 365일 한 번에. 평가일은 마스크(`MASK_COLS`)로 가린 날을 뺀 날이고, **전체 날**과 **특보일+3일**로 나눠 MAPE를 유형별·역별로 봅니다. (나)의 A, B_lag, B, C는 2025년 실제 특보·날씨를 입력으로 준 **사후 상한선**입니다.
3. **2024 validation과 나란히 비교**: 모형 순위와 단계별 개선(Naive → N → A/B)이 2025에서도 유지되는지.
4. **이벤트 기준선 (2025)**: 깨끗한 단일 이벤트에 04번과 같은 절차(지속일수+14일 예측, k=1, m=2)를 적용해 이벤트 × 역 표로 **서술적으로만** 보여 주고 2014~2024 결과와 방향을 비교합니다 (이벤트가 적어 검정하지 않음).

> **2025년 요금 인상(`fare_2025`, 2025-06-28)은 학습 기간(2014~2024)에 없어 추정할 수 없습니다. 그래서 `fare_2025` 더미는 모형에서 빼고, 하루 앞 롤링 예측은 수준 변화를 전날 자료로 따라가며 (365일 한 번에 방식은 따라가지 못합니다).** `climate_card`(2024-01-27)는 2024년 자료로 이번에는 추정됩니다.

**비교 기준 표시**: 2024 validation 값은 03·05번이 저장한 2024 일별 예측(`model_compare_pred2024.csv`, `sarimax_NABC_pred2024.csv`)을 같은 방식으로 다시 채점한 값입니다.

**입력 파일**

- `data/processed/model_input_9stations.csv` (2025년 행 포함), `results/events.csv`, `results/event_summary_by_type.csv`, `results/event_effects.csv` (2014~2024 결과), `results/model_compare_pred2024.csv`, `results/sarimax_NABC_pred2024.csv`

**출력 파일**

- `results/test2025_by_station.csv`, `test2025_by_type.csv`, `test2025_vs_valid2024.csv`, `test2025_steps.csv`, `test2025_pred.csv`, `test2025_fits.csv`
- `results/test2025_event_effects.csv`, `test2025_event_by_station_maxdrop.csv`, `test2025_event_by_station_recovery.csv`, `test2025_event_summary.csv`, `test2025_event_vs_2014_2024.csv`, `test2025_event_daily_paths.csv`
- `results/test2025_run_info.json` (실행 일시와 설정 기록)


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다.

In [2]:
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import statsmodels
from joblib import Parallel, delayed
from scipy.stats import kendalltau
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

**한 번만 실행하는 안전장치**: 실행 기록 파일(`test2025_run_info.json`)이 이미 있으면 이 노트북은 실행되지 않습니다. 2025년 test는 한 번만 돌리고, 결과를 보고 설정을 바꿔 다시 돌리지 않기 위해서입니다.

In [3]:
RUN_INFO = RESULTS / 'test2025_run_info.json'
if RUN_INFO.exists():
    raise RuntimeError('2025 test는 이미 한 번 실행했습니다 (test2025_run_info.json). 결과를 보고 설정을 바꿔 다시 돌리면 안 되므로 다시 실행하지 않습니다.')
print('실행 기록 파일이 없음: 이번이 첫 실행입니다.')

실행 기록 파일이 없음: 이번이 첫 실행입니다.


**test 연도와 재학습 기간**: test 연도는 `00_settings`의 `TEST`(2025)입니다. 재학습은 test 전해 끝(2024-12-31)까지이고, 평가 구간은 test 연도 전체입니다. `EVENT_PERIOD`는 02번 이벤트 표에서 이 test 연도의 이벤트를 부르는 기간 이름입니다.

In [4]:
TEST_YEAR = TEST                                                      # 2025 (설정 고정)
RETRAIN_END = f'{TEST_YEAR - 1}-12-31'                                # 2014~2024로 다시 학습 (2024 validation 연도까지 포함)
TEST_START, TEST_END = pd.Timestamp(f'{TEST_YEAR}-01-01'), pd.Timestamp(f'{TEST_YEAR}-12-31')
EVENT_PERIOD = {2025: 'test', 2024: 'validation'}[TEST_YEAR]
MODELS = ['Naive', 'N', 'A', 'B_lag', 'B', 'C']                       # 평가할 모형
MODES = ['(가) 하루 앞 롤링', '(나) 365일 한 번에']
SUBS = ['전체', '특보일+3일']
DATA = PROC / 'model_input_9stations.csv'
print('재학습: 2014-01-01 ~', RETRAIN_END, '| 평가:', TEST_START.date(), '~', TEST_END.date())

재학습: 2014-01-01 ~ 2024-12-31 | 평가: 2025-01-01 ~ 2025-12-31


아래 함수들은 `03_sarimax`·`05_model_compare`와 **같은 코드**입니다 (입력 만들기, 적합과 미수렴 재시도, 하루 앞 롤링 예측). 학습 끝·평가 구간 이름만 이 노트북용(`RETRAIN_END`, `TEST_START`, `TEST_END`)으로 바꿨습니다.

**`load`**: `model_input_9stations.csv`를 읽습니다 (test 연도 끝까지).

In [5]:
def load():
    d = pd.read_csv(DATA, parse_dates=['date'])
    return d[d.date <= TEST_END]                              # 2025(test)는 여기서 버림

**`prepare`**: 한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.

In [6]:
def prepare(d, station, season=False, model='B'):
    """한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.
    model: 'B' = ASOS 실측만, 'A' = 특보만, 'C' = 특보 + ASOS, 'N' = 날씨 없음, 'B_lag'/'C_lag' = B/C에서 당일 ASOS를 뺀 버전"""
    s = d[d.station == station].set_index('date').sort_index().asfreq('D')
    assert s.index.min() == START and s.index.max() == TEST_END and s.station.notna().all(), '날짜 빈틈'
    y = np.log(s.total)
    y[s[MASK_COLS].sum(axis=1) > 0] = np.nan                 # 가리기: 날짜는 유지, y만 NaN
    blocks = []
    if model in ('B', 'C', 'B_lag', 'C_lag'):
        w = pd.DataFrame({
            'heat_excess': (s.feels_like_max - HEAT_BASE).clip(lower=0).fillna(0),
            'cold_excess': (COLD_BASE - s.tmin).clip(lower=0).fillna(0),
            'rain': np.log1p(s.rain_mm),
            'snow': s.snow_new_cm,
        }).fillna(0)
        blocks += ([] if model.endswith('_lag') else [w]) + [pd.DataFrame({f'{c}_lag{k}': w[c].shift(k).fillna(0) for c in WEATHER for k in LAGS})]
    if model in ('A', 'C', 'C_lag'):
        a = s[ALERTS].fillna(0)                                 # NaN은 2014-01-01~07뿐(y도 가려짐)
        blocks += [a, pd.DataFrame({f'{c}_lag{k}': a[c].shift(k).fillna(0) for c in ALERTS for k in LAGS})]
    steps = list(STEP_COLS) + (list(SEASON_COLS) if season else []) + ([STATION_STEP[station]] if station in STATION_STEP else [])
    X = pd.concat([s[steps]] + blocks, axis=1).astype(float)
    return y, X, s.group.iloc[0], s.total

**`_fit`**: SARIMAX를 한 번 적합합니다(경고를 모아 둠).

In [7]:
def _fit(model, start=None, maxiter=None, method=None, **kw):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        res = model.fit(start_params=start, disp=False, method=method or FIT_METHOD, maxiter=maxiter or FIT_MAXITER, **kw)
    return res, sorted({f'{w.category.__name__}: {str(w.message)[:110]}' for w in caught})

**`_converged`**: 적합이 수렴했는지 확인합니다.

In [8]:
def _converged(res):
    return bool(res.mle_retvals.get('converged'))

**`alt_start`**: 다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값

In [9]:
def alt_start(model, ytr, Xtr):
    """다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값"""
    dy, dX = ytr - ytr.shift(7), Xtr - Xtr.shift(7)
    ok = dy.notna() & dX.notna().all(axis=1)
    beta = np.linalg.lstsq(dX[ok].values, dy[ok].values, rcond=None)[0]
    sig2 = float(np.var(dy[ok].values - dX[ok].values @ beta))
    cols, out = list(Xtr.columns), []
    for n in model.param_names:
        if n in cols: out.append(beta[cols.index(n)])
        elif n.startswith('ar.S'): out.append(0.1)
        elif n.startswith('ma.S'): out.append(-0.8)
        elif n == 'ar.L1': out.append(0.5)
        elif n.startswith(('ar.', 'ma.')): out.append(0.0)
        else: out.append(sig2)
    return np.array(out)

**`fit_station`**: 2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.

In [10]:
def fit_station(y, X, order, seasonal_order, retry=False, hint=None):
    """2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.
    retry=True면 기본 설정(lbfgs, maxiter=1000)이 수렴하지 않을 때 순서대로 다시 시도:
    (1) 기본 결과의 계수에서 이어서 maxiter=5000, (2) 다른 초기값에서 maxiter=5000. 첫 수렴 결과를 쓰고, 없으면 loglike가 가장 큰 것을 씀.
    hint(다른 모형의 계수 {이름: 값})를 주면 그 계수를 초기값으로 lbfgs(maxiter=5000)를 먼저 돌리고, 수렴하지 않으면
    그 초기값에서 method='nm'을 먼저 돌린 뒤 그 결과로 lbfgs(maxiter=5000)를 다시 돌린다(없는 변수는 0). 그래도 안 되면 미수렴 표시"""
    ytr, Xtr = y[:RETRAIN_END], X[:RETRAIN_END]
    dropped = [c for c in X.columns if (Xtr[c] == 0).all()]
    X = X.drop(columns=dropped); Xtr = Xtr.drop(columns=dropped)
    model = SARIMAX(ytr, exog=Xtr, order=order, seasonal_order=seasonal_order)
    if hint is not None:
        start = np.array([hint.get(n, 0.0) for n in model.param_names])
        r1, m1 = _fit(model, start=start, maxiter=5000)
        tried = [('B 계수를 초기값으로(lbfgs, maxiter=5000)', r1, m1)]
        if not _converged(r1):
            rn, _ = _fit(model, start=start, method='nm', maxiter=3000, maxfun=6000)
            r2, m2 = _fit(model, start=rn.params.values, maxiter=5000)
            tried.append(('B 계수 초기값, nm 후 lbfgs(maxiter=5000)', r2, m2))
    else:
        res, msgs = _fit(model)
        tried = [('기본(lbfgs, maxiter=1000)', res, msgs)]
        if retry and not _converged(res):
            r1, m1 = _fit(model, start=res.params.values, maxiter=5000)
            tried.append(('기본 결과에서 이어서(maxiter=5000)', r1, m1))
            if not _converged(r1):
                r2, m2 = _fit(model, start=alt_start(model, ytr, Xtr), maxiter=5000)
                tried.append(('다른 초기값(차분 회귀, maxiter=5000)', r2, m2))
    best = next((t for t in tried if _converged(t[1])), None) or max(tried, key=lambda t: t[1].llf)
    note = best[0] + ('' if _converged(best[1]) else ' [미수렴, loglike 최대 결과 사용]') + ('' if len(tried) == 1 else f' (시도 {len(tried)}회)')
    return best[1], best[2], dropped, X, note

**`alert_window`**: any_alert=1인 날과 그다음 3일 (test 연도 구간, 불리언 Series).

In [11]:
def alert_window(d, st):
    """any_alert=1인 날과 그다음 3일 (2024년 구간, 불리언 Series)"""
    a = d[d.station == st].set_index('date').sort_index().asfreq('D').any_alert.fillna(0)
    return ((a + a.shift(1) + a.shift(2) + a.shift(3)).fillna(0) > 0)[TEST_START:TEST_END]

**`validate`**: 계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).

In [12]:
def validate(res, y, X, total, window=None):
    """계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).
    window를 주면 그 날들(특보일+3일) 중 가리지 않은 날만으로도 계산"""
    full = res.apply(y, exog=X, refit=False)                    # 2014~2024 전체에 같은 계수 적용(2024 y는 전날까지만 예측에 쓰임)
    pred = np.exp(full.get_prediction(start=TEST_START, end=TEST_END, dynamic=False).predicted_mean)
    act = total[TEST_START:TEST_END]
    ok = y[TEST_START:TEST_END].notna()                      # 가리기된 날 제외
    err = act[ok] - pred[ok]
    naive = total.shift(7)[TEST_START:TEST_END][ok]            # 참고: 7일 전 같은 요일 값
    out = {'n_days': int(ok.sum()), 'n_masked': int((~ok).sum()),
           'RMSE': float(np.sqrt((err ** 2).mean())), 'MAPE': float((err.abs() / act[ok]).mean() * 100),
           'naive7_RMSE': float(np.sqrt(((act[ok] - naive) ** 2).mean())), 'naive7_MAPE': float(((act[ok] - naive).abs() / act[ok]).mean() * 100)}
    detail = pd.DataFrame({'actual': act, 'pred': pred, 'ok': ok})
    if window is not None:
        m = ok & window
        e = act[m] - pred[m]
        out.update(n_alert=int(m.sum()), RMSE_alert=float(np.sqrt((e ** 2).mean())), MAPE_alert=float((e.abs() / act[m]).mean() * 100))
        detail['window'] = window
    out['detail'] = detail
    return out

**`naive_station`**: 팀원 `Naive_SARIMA.ipynb` 방식입니다. 이용량(원 단위)에서 코로나 기간만 NaN으로 두고, 팀원이 고른 역별 차수(`NAIVE_ORDERS`)로 **2014~2024에 다시 적합**한 뒤 ① test 연도 전체를 한 번에 예측하고 ② 같은 계수로 하루 앞 예측합니다.

In [13]:
def naive_station(d, st):
    s = d[d.station == st].set_index('date').sort_index().asfreq('D')
    y = s.total.copy()
    y[NAIVE_COVID[0]:NAIVE_COVID[1]] = np.nan                           # 코로나 기간만 NaN
    order, seasonal = NAIVE_ORDERS[st]
    fitted = SARIMAX(y[:RETRAIN_END], order=order, seasonal_order=seasonal, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
    idx = y[TEST_START:TEST_END].index
    once = fitted.get_forecast(steps=len(idx)).predicted_mean               # 2024-12-31까지의 자료로 test 연도를 한 번에
    once.index = idx
    roll = fitted.apply(y, refit=False).get_prediction(start=TEST_START, end=TEST_END, dynamic=False).predicted_mean   # 하루 앞
    return once, roll

**`score`**: 평가일(`mask`)만으로 원 단위 MAE, RMSE, MAPE(%)를 계산합니다.

In [14]:
def score(actual, pred, mask):
    a, p = actual[mask], pred[mask]
    e = a - p
    return {'n_days': int(mask.sum()), 'MAE': float(e.abs().mean()), 'RMSE': float(np.sqrt((e ** 2).mean())), 'MAPE': float((e.abs() / a).mean() * 100)}

**`station_task`**: 한 역에서 6개 모형을 모두 **2014~2024로 다시 적합**하고(학습 기간에 전부 0인 더미는 자동으로 뺌, 미수렴이면 이어서 다시, C는 수유·연신내에서 B 계수를 초기값으로), (가) 하루 앞 롤링과 (나) 365일 한 번에 예측을 만듭니다. 평가일은 마스크로 가린 날을 뺀 날이고, **전체**와 **특보일+3일**(any_alert=1인 날과 그다음 3일)로 나눠 점수를 냅니다. (나)의 A, B_lag, B, C는 test 연도 실제 특보·날씨를 입력으로 주는 사후 평가입니다.

In [15]:
def station_task(st):
    warnings.filterwarnings('ignore')
    d = load()
    y0, _, grp, total = prepare(d, st, True, 'N')
    ok = y0[TEST_START:TEST_END].notna()                                    # 평가일: 마스크로 가린 날 제외
    win = alert_window(d, st).reindex(ok.index).fillna(False)               # 특보일 + 그다음 3일
    act = total[TEST_START:TEST_END]
    rows, dets, fits = [], [], []
    def add(model, mode, pred):
        pred = pred.reindex(act.index)
        for sub, mask in [(SUBS[0], ok), (SUBS[1], ok & win)]:
            rows.append({'역': st, '유형': grp, '모형': model, '방식': mode, '구간': sub, **score(act, pred, mask)})
        dets.append(pd.DataFrame({'역': st, '모형': model, '방식': mode, 'date': act.index, 'actual': act.values, 'pred': pred.values, 'ok': ok.values, 'window': win.values}))
    once, roll = naive_station(d, st)
    add('Naive', MODES[0], roll)
    add('Naive', MODES[1], once)
    b_params = None
    for m in ['N', 'A', 'B_lag', 'B', 'C']:
        y, X, _, tot = prepare(d, st, True, m)
        hint = b_params if (m == 'C' and st in RETRY_C_FROM_B) else None      # 03번과 같은 규칙: 수유·연신내의 C는 B 계수로 시작
        res, msgs, dropped, Xu, note = fit_station(y, X, ORDER, SEASONAL_ORDER, True, hint)
        if m == 'B':
            b_params = res.params.to_dict()
        v = validate(res, y, Xu, tot)                                       # (가) 하루 앞 롤링
        add(m, MODES[0], v['detail']['pred'])
        full = res.apply(y, exog=Xu, refit=False)
        add(m, MODES[1], np.exp(full.get_prediction(start=TEST_START, end=TEST_END, dynamic=0).predicted_mean))   # (나) 한 번에
        fits.append({'역': st, '모형': m, '수렴': bool(res.mle_retvals.get('converged')), '적합': note, '뺀 더미': ','.join(dropped)})
    return rows, pd.concat(dets, ignore_index=True), fits

**9개 역 병렬 실행**: 역마다 6개 모형을 2014~2024로 다시 적합하고 test 연도를 예측합니다 (약 10분).

In [16]:
out = Parallel(n_jobs=WORKERS)(delayed(station_task)(st) for st in STATIONS)
res_all = pd.DataFrame([r for o in out for r in o[0]])
pred_all = pd.concat([o[1] for o in out], ignore_index=True)
fits = pd.DataFrame([r for o in out for r in o[2]])
RESULTS.mkdir(exist_ok=True)
res_all.to_csv(RESULTS / 'test2025_by_station.csv', index=False, encoding='utf-8-sig')
pred_all.to_csv(RESULTS / 'test2025_pred.csv', index=False, encoding='utf-8-sig')
fits.to_csv(RESULTS / 'test2025_fits.csv', index=False, encoding='utf-8-sig')
print('지표 행', res_all.shape, '| 일별 예측', pred_all.shape, '| 미수렴 적합:', int((~fits.수렴).sum()), '/', len(fits))
print('학습 기간에 값이 전부 0이라 뺀 더미:', sorted(set(','.join(fits['뺀 더미']).split(',')) - {''}))
fits[~fits.수렴]

지표 행 (216, 9) | 일별 예측 (39420, 8) | 미수렴 적합: 0 / 45
학습 기간에 값이 전부 0이라 뺀 더미: ['fare_2025']


,역,모형,수렴,적합,뺀 더미


**재학습 적합 요약**: 모형별 수렴 여부와 적합 방법을 확인합니다 (기본이 아닌 적합 방법만 보여 줌).

In [17]:
fits[fits.적합 != '기본(lbfgs, maxiter=1000)'][['역', '모형', '수렴', '적합']].reset_index(drop=True)

,역,모형,수렴,적합
0,시청,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
1,역삼,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
2,선릉,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
3,신촌,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
4,혜화,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
5,신사,B,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
6,신사,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
7,서울대입구,B_lag,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
8,서울대입구,C,True,기본 결과에서 이어서(maxiter=5000) (시도 2회)
9,수유,C,True,"B 계수를 초기값으로(lbfgs, maxiter=5000)"


## 2025년 예측 평가

**역별 MAPE (%)**: 유형 순서로 역을 놓고, 방식과 구간(전체 / 특보일+3일)별로 6개 모형을 나란히 봅니다. 평가일은 마스크로 가린 날을 뺀 날입니다 (역별 일수는 아래에 표시).

In [18]:
st_order = [s for g in TYPE_ORDER for s in STATIONS if res_all[res_all.역 == s].유형.iloc[0] == g]
res_all['역'] = pd.Categorical(res_all.역, st_order, ordered=True)
print('평가일 수(역별, 전체 / 특보일+3일):', {s: (int(res_all[(res_all.역 == s) & (res_all.구간 == SUBS[0])].n_days.iloc[0]), int(res_all[(res_all.역 == s) & (res_all.구간 == SUBS[1])].n_days.iloc[0])) for s in st_order})
by_st = res_all.pivot_table(index=['유형', '역'], columns=['방식', '구간', '모형'], values='MAPE', observed=True)
by_st = by_st.reindex(columns=pd.MultiIndex.from_product([MODES, SUBS, MODELS]))
by_st.round(2)

평가일 수(역별, 전체 / 특보일+3일): {'시청': (330, 128), '역삼': (343, 130), '선릉': (343, 130), '신촌': (343, 130), '혜화': (343, 130), '신사': (343, 130), '서울대입구': (343, 130), '수유': (343, 130), '연신내': (343, 130)}


(가) 하루 앞 롤링                                \
                                      전체                                 
                                   Naive     N     A B_lag     B     C   
유형                   역                                                   
Commercial & Leisure 신촌     4.490000e+00  3.36  3.33  3.38  3.24  3.25   
                     혜화     1.040259e+12  3.89  3.95  3.98  3.88  3.88   
                     신사     4.810000e+00  2.95  2.96  2.96  2.92  2.91   
Employment           시청     7.460000e+00  4.83  4.85  4.98  4.94  4.95   
                     역삼     6.940000e+00  2.90  2.95  2.96  2.99  2.99   
                     선릉     5.840000e+00  2.51  2.51  2.52  2.50  2.51   
Residential          서울대입구  3.800000e+00  2.31  2.28  2.36  2.14  2.13   
                     수유     3.510000e+00  2.26  2.21  2.33  2.00  1.96   
                     연신내    3.690000e+00  2.79  2.73  2.87  2.49  2.45   

                                                            ... (나) 365일 한 번에  \
                                  특보일+3일                    ...            전체   
                                   Naive     N     A B_lag  ...             A   
유형                   역                                      ...                 
Commercial & Leisure 신촌     3.730000e+00  2.73  2.64  2.81  ...          5.83   
                     혜화     9.386541e+11  2.85  3.00  3.13  ...          5.69   
                     신사     4.820000e+00  3.14  3.15  3.27  ...          3.82   
Employment           시청     7.710000e+00  5.29  5.39  5.72  ...          6.67   
                     역삼     6.910000e+00  2.78  2.85  2.95  ...          4.44   
                     선릉     5.500000e+00  2.34  2.40  2.48  ...          4.00   
Residential          서울대입구  3.470000e+00  2.16  2.10  2.33  ...          3.49   
                     수유     3.260000e+00  2.14  2.03  2.37  ...          3.51   
                     연신내    3.320000e+00  2.66  2.58  2.90  ...          6.65   

                                                                              \
                                                    특보일+3일                     
                           B_lag     B     C         Naive     N     A B_lag   
유형                   역                                                         
Commercial & Leisure 신촌     5.81  5.62  5.60  7.930000e+00  6.89  6.50  6.38   
                     혜화     6.12  5.74  5.52  9.386541e+11  5.80  4.80  5.75   
                     신사     4.06  3.77  3.76  6.310000e+00  4.75  4.18  4.81   
Employment           시청     6.82  6.81  6.79  1.210000e+01  7.04  7.04  7.18   
                     역삼     4.44  4.58  4.56  6.540000e+00  4.02  4.30  4.32   
                     선릉     4.00  4.17  4.18  5.830000e+00  3.16  3.61  3.49   
Residential          서울대입구  3.59  3.53  3.43  4.150000e+00  3.22  2.89  3.15   
                     수유     3.64  3.44  3.32  7.190000e+00  3.97  3.34  3.71   
                     연신내    6.64  6.86  6.88  3.930000e+00  4.78  5.22  5.25   

                                        
                                        
                               B     C  
유형                   역                  
Commercial & Leisure 신촌     6.16  6.11  
                     혜화     5.11  4.71  
                     신사     4.40  4.29  
Employment           시청     7.17  7.32  
                     역삼     4.48  4.53  
                     선릉     4.00  4.08  
Residential          서울대입구  3.19  3.00  
                     수유     3.29  3.04  
                     연신내    5.73  5.83  

[9 rows x 24 columns]

**유형별 평균**: 역별 지표를 유형별로 평균합니다 (MAPE, MAE, RMSE). 전체(9개 역) 평균도 함께 저장합니다. 365일 한 번에의 A·B_lag·B·C는 **사후 상한선**입니다.

In [19]:
res_all['역'] = res_all.역.astype(str)
mt_cols = ['n_days', 'MAE', 'RMSE', 'MAPE']
by_type = res_all.groupby(['방식', '구간', '유형', '모형'])[mt_cols].mean().reset_index()
tot = res_all.groupby(['방식', '구간', '모형'])[mt_cols].mean().reset_index()
tot.insert(2, '유형', '전체(9개 역)')
by_type = pd.concat([by_type, tot], ignore_index=True)
by_type['post_hoc'] = by_type.방식.eq(MODES[1]) & by_type.모형.isin(['A', 'B_lag', 'B', 'C'])        # 365일 한 번에의 외생변수 모형 = 사후 상한선
by_type['모형'] = pd.Categorical(by_type.모형, MODELS, ordered=True)
by_type = by_type.sort_values(['방식', '구간', '유형', '모형']).reset_index(drop=True)
by_type.to_csv(RESULTS / 'test2025_by_type.csv', index=False, encoding='utf-8-sig')
by_type.pivot_table(index=['방식', '구간', '유형'], columns='모형', values='MAPE', observed=True).reindex(columns=MODELS).round(2)

모형                                                Naive     N     A  B_lag  \
방식            구간     유형                                                      
(가) 하루 앞 롤링   전체     Commercial & Leisure  3.467529e+11  3.40  3.41   3.44   
                     Employment            6.750000e+00  3.41  3.44   3.49   
                     Residential           3.660000e+00  2.45  2.41   2.52   
                     전체(9개 역)              1.155843e+11  3.09  3.08   3.15   
              특보일+3일 Commercial & Leisure  3.128847e+11  2.91  2.93   3.07   
                     Employment            6.710000e+00  3.47  3.55   3.72   
                     Residential           3.350000e+00  2.32  2.24   2.53   
                     전체(9개 역)              1.042949e+11  2.90  2.90   3.11   
(나) 365일 한 번에 전체     Commercial & Leisure  3.467529e+11  5.38  5.11   5.33   
                     Employment            7.620000e+00  5.11  5.04   5.09   
                     Residential           4.910000e+00  4.69  4.55   4.63   
                     전체(9개 역)              1.155843e+11  5.06  4.90   5.01   
              특보일+3일 Commercial & Leisure  3.128847e+11  5.82  5.16   5.64   
                     Employment            8.160000e+00  4.74  4.98   5.00   
                     Residential           5.090000e+00  3.99  3.82   4.04   
                     전체(9개 역)              1.042949e+11  4.85  4.65   4.89   

모형                                            B     C  
방식            구간     유형                                
(가) 하루 앞 롤링   전체     Commercial & Leisure  3.35  3.34  
                     Employment            3.48  3.48  
                     Residential           2.21  2.18  
                     전체(9개 역)              3.01  3.00  
              특보일+3일 Commercial & Leisure  3.08  3.06  
                     Employment            3.74  3.74  
                     Residential           2.35  2.28  
                     전체(9개 역)              3.06  3.03  
(나) 365일 한 번에 전체     Commercial & Leisure  5.04  4.96  
                     Employment            5.19  5.18  
                     Residential           4.61  4.54  
                     전체(9개 역)              4.95  4.89  
              특보일+3일 Commercial & Leisure  5.22  5.04  
                     Employment            5.21  5.31  
                     Residential           4.07  3.96  
                     전체(9개 역)              4.83  4.77

## 2024 validation과 나란히 비교

**2024 validation 점수 다시 채점**: 03·05번이 저장한 2024 일별 예측을 같은 방식(평가일 = 마스크로 가린 날 제외, 전체 / 특보일+3일)으로 채점합니다. C의 (나) 365일 한 번에는 2024 결과가 저장돼 있지 않아 비워 둡니다.

In [20]:
p1 = pd.read_csv(RESULTS / 'model_compare_pred2024.csv', parse_dates=['date'])        # Naive, N, A, B_lag, B (두 방식)
p2 = pd.read_csv(RESULTS / 'sarimax_NABC_pred2024.csv', parse_dates=['date'])         # C 포함 (하루 앞 롤링만)
d_all = load()
V0, V1 = pd.Timestamp('2024-01-01'), pd.Timestamp('2024-12-31')
def window_2024(st):
    a = d_all[d_all.station == st].set_index('date').sort_index().asfreq('D').any_alert.fillna(0)
    return ((a + a.shift(1) + a.shift(2) + a.shift(3)).fillna(0) > 0)[V0:V1]
g_of = d_all.drop_duplicates('station').set_index('station').group
rows24 = []
parts = [(p1, None), (p2[p2.모형 == 'C'].assign(방식=MODES[0]), None)]
for pdf, _ in parts:
    for (st, m, mode), g in pdf.groupby(['역', '모형', '방식']):
        g = g.set_index('date').sort_index()
        win = window_2024(st).reindex(g.index).fillna(False)
        for sub, mask in [(SUBS[0], g.ok), (SUBS[1], g.ok & win)]:
            rows24.append({'역': st, '유형': g_of[st], '모형': m, '방식': mode, '구간': sub, **score(g.actual, g.pred, mask)})
res24 = pd.DataFrame(rows24)
print('2024 validation 재채점', res24.shape, '| 모형:', sorted(res24.모형.unique()))

2024 validation 재채점 (198, 9) | 모형: ['A', 'B', 'B_lag', 'C', 'N', 'Naive']


**2024 vs 2025 모형 순위와 MAPE**: 방식·구간·유형별로 모형의 MAPE와 순위(1 = 가장 낮음)를 2024와 2025에 나란히 놓습니다. 순위 상관(Kendall tau)이 1에 가까우면 순위가 유지된 것입니다. `results/test2025_vs_valid2024.csv`로 저장합니다.

In [21]:
def type_means(df):
    a = df.groupby(['방식', '구간', '유형', '모형']).MAPE.mean()
    b = df.groupby(['방식', '구간', '모형']).MAPE.mean().to_frame().assign(유형='전체(9개 역)').set_index('유형', append=True).reorder_levels(['방식', '구간', '유형', '모형'])
    return pd.concat([a.to_frame(), b.rename(columns={'MAPE': 'MAPE'})]).MAPE
m24, mtest = type_means(res24), type_means(res_all)
cmp_ = pd.concat({'MAPE_2024': m24, f'MAPE_{TEST_YEAR}': mtest}, axis=1).reset_index()
cmp_['모형'] = pd.Categorical(cmp_.모형, MODELS, ordered=True)
cmp_ = cmp_.sort_values(['방식', '구간', '유형', '모형']).reset_index(drop=True)
cmp_['순위_2024'] = cmp_.groupby(['방식', '구간', '유형']).MAPE_2024.rank(method='min')
cmp_[f'순위_{TEST_YEAR}'] = cmp_.groupby(['방식', '구간', '유형'])[f'MAPE_{TEST_YEAR}'].rank(method='min')
cmp_['post_hoc'] = cmp_.방식.eq(MODES[1]) & cmp_.모형.isin(['A', 'B_lag', 'B', 'C'])
cmp_.to_csv(RESULTS / 'test2025_vs_valid2024.csv', index=False, encoding='utf-8-sig')
rk = []
for (mode, sub, typ), g in cmp_.groupby(['방식', '구간', '유형']):
    g = g.dropna(subset=['MAPE_2024', f'MAPE_{TEST_YEAR}'])
    tau = kendalltau(g.MAPE_2024, g[f'MAPE_{TEST_YEAR}']).statistic if len(g) > 2 else np.nan
    rk.append({'방식': mode, '구간': sub, '유형': typ, '2024 순서(좋은 순)': ' < '.join(g.sort_values('MAPE_2024').모형.astype(str)),
               f'{TEST_YEAR} 순서(좋은 순)': ' < '.join(g.sort_values(f'MAPE_{TEST_YEAR}').모형.astype(str)), 'Kendall tau': round(tau, 2)})
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rk)

,방식,구간,유형,2024 순서(좋은 순),2025 순서(좋은 순),Kendall tau
0,(가) 하루 앞 롤링,전체,Commercial & Leisure,B < A < N < C < B_lag < Naive,C < B < N < A < B_lag < Naive,0.47
1,(가) 하루 앞 롤링,전체,Employment,A < B < C < B_lag < N < Naive,N < A < B < C < B_lag < Naive,0.47
2,(가) 하루 앞 롤링,전체,Residential,C < B < A < B_lag < N < Naive,C < B < A < N < B_lag < Naive,0.87
3,(가) 하루 앞 롤링,전체,전체(9개 역),B < C < A < N < B_lag < Naive,C < B < A < N < B_lag < Naive,0.87
4,(가) 하루 앞 롤링,특보일+3일,Commercial & Leisure,B < N < A < B_lag < C < Naive,N < A < C < B_lag < B < Naive,0.33
5,(가) 하루 앞 롤링,특보일+3일,Employment,A < B < N < B_lag < C < Naive,N < A < B_lag < B < C < Naive,0.60
6,(가) 하루 앞 롤링,특보일+3일,Residential,A < C < B < B_lag < N < Naive,A < C < N < B < B_lag < Naive,0.73
7,(가) 하루 앞 롤링,특보일+3일,전체(9개 역),A < B < C < B_lag < N < Naive,N < A < C < B < B_lag < Naive,0.33
8,(나) 365일 한 번에,전체,Commercial & Leisure,B < A < N < B_lag < Naive,B < A < B_lag < N < Naive,0.80
9,(나) 365일 한 번에,전체,Employment,B_lag < B < A < N < Naive,A < B_lag < N < B < Naive,0.40


**역별 2024 vs 2025 (하루 앞 롤링, 전체 날)**: 같은 방식으로 역마다 MAPE를 2024와 2025에 나란히 봅니다.

In [22]:
sv = pd.concat({'2024': res24, str(TEST_YEAR): res_all}, names=['연도']).reset_index(level=0)
sv = sv[(sv.방식 == MODES[0]) & (sv.구간 == SUBS[0])]
sv_t = sv.pivot_table(index=['유형', '역'], columns=['연도', '모형'], values='MAPE', observed=True)
sv_t = sv_t.reindex(columns=pd.MultiIndex.from_product([['2024', str(TEST_YEAR)], MODELS]))
sv_t = sv_t.loc[[(g, s) for g in TYPE_ORDER for s in STATIONS if g_of[s] == g]]
pd.concat({'2024': res24, str(TEST_YEAR): res_all}, names=['연도']).reset_index(level=0).to_csv(RESULTS / 'test2025_station_vs_2024.csv', index=False, encoding='utf-8-sig')
sv_t.round(2)

2024                                        2025  \
                           Naive     N     A B_lag     B     C         Naive   
유형                   역                                                         
Employment           시청     7.06  5.21  5.06  5.14  5.01  5.10  7.460000e+00   
                     역삼     7.01  2.50  2.50  2.52  2.57  2.58  6.940000e+00   
                     선릉     6.22  2.49  2.45  2.51  2.46  2.44  5.840000e+00   
Commercial & Leisure 신촌     4.60  3.16  3.16  3.22  3.13  3.22  4.490000e+00   
                     혜화     4.24  3.66  3.69  3.71  3.71  3.76  1.040259e+12   
                     신사     5.23  2.68  2.65  2.70  2.61  2.59  4.810000e+00   
Residential          서울대입구  4.07  2.13  2.07  2.13  2.02  2.04  3.800000e+00   
                     수유     3.80  2.14  2.09  2.13  1.98  1.96  3.510000e+00   
                     연신내    4.02  2.37  2.26  2.30  2.01  1.97  3.690000e+00   

                                                          
                               N     A B_lag     B     C  
유형                   역                                    
Employment           시청     4.83  4.85  4.98  4.94  4.95  
                     역삼     2.90  2.95  2.96  2.99  2.99  
                     선릉     2.51  2.51  2.52  2.50  2.51  
Commercial & Leisure 신촌     3.36  3.33  3.38  3.24  3.25  
                     혜화     3.89  3.95  3.98  3.88  3.88  
                     신사     2.95  2.96  2.96  2.92  2.91  
Residential          서울대입구  2.31  2.28  2.36  2.14  2.13  
                     수유     2.26  2.21  2.33  2.00  1.96  
                     연신내    2.79  2.73  2.87  2.49  2.45

**단계별 개선 (Naive → N → A / B)**: 유형별 평균 MAPE에서 ① Naive → N(마스크·달력 더미·log), ② N → A(특보), ③ N → B_lag(전날까지의 날씨), ④ N → B(당일 날씨), ⑤ N → C의 개선(%p, 양수 = MAPE가 줄어듦)을 2024와 2025에 나란히 정리합니다. `results/test2025_steps.csv`로 저장합니다.

In [23]:
def steps(series):
    mt = series.unstack('모형')[MODELS]
    s = pd.DataFrame(index=mt.index)
    s['Naive→N (%p)'] = mt['Naive'] - mt['N']
    s['Naive→N (%)'] = (mt['Naive'] - mt['N']) / mt['Naive'] * 100
    for m in ['A', 'B_lag', 'B', 'C']:
        s[f'N→{m} (%p)'] = mt['N'] - mt[m]
        s[f'N→{m} (%)'] = (mt['N'] - mt[m]) / mt['N'] * 100
    return s
st24, sttest = steps(m24), steps(mtest)
steps_cmp = pd.concat({'2024': st24, str(TEST_YEAR): sttest}, axis=1).round(2)
steps_cmp.to_csv(RESULTS / 'test2025_steps.csv', encoding='utf-8-sig')
key = ['Naive→N (%p)', 'Naive→N (%)', 'N→A (%p)', 'N→B_lag (%p)', 'N→B (%p)', 'N→C (%p)']
steps_cmp.loc[[ix for ix in steps_cmp.index if ix[2] == '전체(9개 역)'], [(y, k) for k in key for y in ['2024', str(TEST_YEAR)]]]

2024          2025        2024  \
                              Naive→N (%p)  Naive→N (%p) Naive→N (%)   
방식            구간     유형                                                
(가) 하루 앞 롤링   전체     전체(9개 역)         2.21  1.155843e+11       43.06   
              특보일+3일 전체(9개 역)         1.43  1.042949e+11       32.21   
(나) 365일 한 번에 전체     전체(9개 역)         0.94  1.155843e+11       15.99   
              특보일+3일 전체(9개 역)         0.42  1.042949e+11        6.21   

                                     2025     2024     2025         2024  \
                              Naive→N (%) N→A (%p) N→A (%p) N→B_lag (%p)   
방식            구간     유형                                                    
(가) 하루 앞 롤링   전체     전체(9개 역)       100.0     0.05     0.00        -0.00   
              특보일+3일 전체(9개 역)       100.0     0.08    -0.00         0.02   
(나) 365일 한 번에 전체     전체(9개 역)       100.0     0.29     0.16         0.16   
              특보일+3일 전체(9개 역)       100.0     0.84     0.19         0.33   

                                      2025     2024     2025     2024     2025  
                              N→B_lag (%p) N→B (%p) N→B (%p) N→C (%p) N→C (%p)  
방식            구간     유형                                                         
(가) 하루 앞 롤링   전체     전체(9개 역)        -0.06     0.09     0.08     0.08     0.08  
              특보일+3일 전체(9개 역)        -0.21     0.06    -0.16     0.03    -0.13  
(나) 365일 한 번에 전체     전체(9개 역)         0.05     0.38     0.11      NaN     0.17  
              특보일+3일 전체(9개 역)        -0.04     0.85     0.01      NaN     0.08

**유형별 단계별 개선 (하루 앞 롤링, 전체 날)**: 위 표 중 하루 앞 롤링의 전체 날만 유형별로 보여 줍니다.

In [24]:
steps_cmp.loc[(MODES[0], SUBS[0]), [(y, k) for k in key for y in ['2024', str(TEST_YEAR)]]]

,2024,2025,2024,2025,2024,2025,2024,2025,2024,2025,2024,2025
,Naive→N (%p),Naive→N (%p),Naive→N (%),Naive→N (%),N→A (%p),N→A (%p),N→B_lag (%p),N→B_lag (%p),N→B (%p),N→B (%p),N→C (%p),N→C (%p)
유형,,,,,,,,,,,,
Commercial & Leisure,1.53,3.467529e+11,32.53,100.00,0.00,-0.01,-0.05,-0.04,0.01,0.05,-0.02,0.06
Employment,3.36,3.340000e+00,49.71,49.42,0.07,-0.02,0.01,-0.07,0.05,-0.06,0.03,-0.07
Residential,1.75,1.210000e+00,44.15,33.12,0.08,0.05,0.03,-0.07,0.21,0.24,0.22,0.27
전체(9개 역),2.21,1.155843e+11,43.06,100.00,0.05,0.00,-0.00,-0.06,0.09,0.08,0.08,0.08


## 이벤트 기준선 (2025)

04번과 **같은 절차**를 2025년 깨끗한 단일 이벤트에 적용합니다. 기준선 모형은 04번과 같은 SARIMA(1,0,1)(1,1,1,7)(계단 더미 + `summer_vacation`, y는 `MASK_COLS`와 특보일+해제 후 7일을 가림)이고 **2014~2024로 다시 학습**한 계수를 고정합니다. 이벤트마다 시작 전날까지의 자료로 지속일수 + 14일을 예측해 95% 구간을 얻고, 충격은 구간 아래인 날이 `SHOCK_K`(1)일 이상, 회복은 구간 안에 `RECOVERY_M`(2)일 연속 머문 첫날입니다 (다음 특보·가림일 전날까지만 확인, 못 돌아오면 중도절단). **이벤트가 적어 이벤트 × 역 표로 서술적으로만 보여 주고 검정하지 않습니다.**

**04번의 이벤트 함수**: `eval_event`(한 이벤트×역의 충격·회복 판정)는 04번과 **같은 코드**입니다.

In [25]:
ONE = pd.Timedelta(days=1)

def eval_event(g, s, e, k, m):
    """g: 날짜 인덱스의 일별 표(actual, base, lower, upper, mask_day, alert_day). s, e: 이벤트 시작·종료일"""
    imp = g.loc[s:e]
    D = len(imp)
    d = imp.actual / imp.base - 1
    n_below = int((imp.actual < imp.lower).sum())
    out = {'지속일수': D, 'n_below': n_below, '충격': n_below >= min(k, D), '최대감소율%': float(-d.min() * 100), '평균편차%': float(d.mean() * 100),
           '기준선평균': float(imp.base.mean()), '실제평균': float(imp.actual.mean()), '누적손실_기간': float((imp.base - imp.actual).sum())}
    post = g.loc[e + ONE:]
    stop = post.index[post.mask_day | post.alert_day]                        # 다음 특보 또는 가림일이 나오면 그 전날까지만 봄
    truncated, gap = '', np.nan
    if len(stop):
        truncated = '다음특보' if bool(post.loc[stop[0], 'alert_day']) else '가림일'
        gap = (stop[0] - e).days                                              # 해제 후 며칠째에 막혔는가
        post = post.loc[:stop[0] - ONE]
    elif len(post) < EVENT_POST_DAYS:
        truncated = '자료끝'
    inside = ((post.actual >= post.lower) & (post.actual <= post.upper)).astype(int)
    rec = None
    if len(post) >= m:
        run = inside.rolling(m).sum().shift(-(m - 1)) == m                   # t부터 m일이 모두 구간 안인가
        idx = run[run].index
        rec = idx[0] if len(idx) else None
    last = (rec - ONE) if rec is not None else (post.index.max() if len(post) else e)
    out.update({'확인창일수': len(post), '확인창제한': truncated, '막힌날(해제후일차)': gap, '회복일': rec, '회복지연일': (rec - e).days if rec is not None else np.nan, '중도절단': rec is None,
                '누적손실_회복까지': float((g.loc[s:last].base - g.loc[s:last].actual).sum())})
    return out

**`event_station_task`**: 04번의 `station_task`와 같은 코드에서 학습 끝(`RETRAIN_END`)과 데이터 끝(`TEST_END`)만 바꿨습니다. 한 역에서 y를 가려 2014~2024로 적합하고 계수를 고정한 뒤 이벤트마다 동적 예측(시작 전날까지의 자료만 사용)을 합니다.

In [26]:
def event_station_task(st, jobs):
    warnings.filterwarnings('ignore')
    d = pd.read_csv(PROC / 'model_input_9stations.csv', parse_dates=['date'])
    d = d[(d.station == st) & (d.date <= TEST_END)]                        # 2025년 행은 읽자마자 버림
    s = d.set_index('date').sort_index().asfreq('D')
    y = np.log(s.total)
    y[s[MASK_COLS].sum(axis=1) > 0] = np.nan                                # 기본 가림
    a = s.any_alert.fillna(0)
    am = a > 0
    for k in range(1, ALERT_MASK_AFTER_DAYS + 1):
        am |= (a.shift(k).fillna(0) > 0)                                    # 특보일과 해제 후 7일
    y[am] = np.nan
    X = s[STEP_COLS + SEASON_COLS + ([STATION_STEP[st]] if st in STATION_STEP else [])].astype(float)
    Xtr = X[:RETRAIN_END]
    drop = [c for c in X.columns if (Xtr[c] == 0).all()]                    # 학습 중 전부 0인 더미(fare_2025, climate_card 등)
    X = X.drop(columns=drop)
    model = SARIMAX(y[:RETRAIN_END], exog=X[:RETRAIN_END], order=ORDER, seasonal_order=SEASONAL_ORDER)
    res = model.fit(disp=False, method=FIT_METHOD, maxiter=FIT_MAXITER)
    if not res.mle_retvals.get('converged'):                                # 미수렴이면 이어서 다시
        res = model.fit(start_params=res.params.values, disp=False, method=FIT_METHOD, maxiter=5000)
    full = res.apply(y, exog=X, refit=False)                                # 계수 고정, 2014~2024 전체에 적용
    mday = (s[MASK_COLS].sum(axis=1) > 0)
    rows = []
    for j in jobs:
        e_last = min(j['start'] + pd.Timedelta(days=j['H'] - 1), TEST_END)  # 2024-12-31을 넘지 않게 자름
        H = (e_last - j['start']).days + 1
        p = full.get_prediction(start=j['start'], end=e_last, dynamic=0)     # 시작 전날까지의 자료만 쓰는 동적 예측
        ci = p.conf_int(alpha=BAND_ALPHA)
        g = pd.DataFrame({'actual': s.total.loc[j['start']:e_last], 'base': np.exp(p.predicted_mean), 'lower': np.exp(ci.iloc[:, 0]), 'upper': np.exp(ci.iloc[:, 1]),
                          'mask_day': mday.loc[j['start']:e_last], 'alert_day': (a > 0).loc[j['start']:e_last]})
        g.insert(0, 'id', j['id']); g.insert(1, '역', st); g.insert(2, 'end', j['end'])
        g['prev_obs60'] = int(y.loc[j['start'] - pd.Timedelta(days=60):j['start'] - pd.Timedelta(days=1)].notna().sum())
        rows.append(g)
    fit = {'역': st, '유형': s.group.iloc[0], 'AIC': res.aic, '수렴': bool(res.mle_retvals.get('converged')), '학습 y 관측일수': int(y[:RETRAIN_END].notna().sum()),
           '학습 일수': len(y[:RETRAIN_END]), '뺀 더미': ','.join(drop)}
    return pd.concat(rows).reset_index().rename(columns={'index': 'date'}), fit

**분석 이벤트 고르기**: 02번 이벤트 표에서 test 연도의 **깨끗한 단일 이벤트**를 고릅니다 (04번의 "단일(주 분석)"과 같은 기준). 이벤트가 적으므로 대설은 참고용입니다.

In [27]:
ev_all = pd.read_csv(RESULTS / 'events.csv', parse_dates=['시작일', '종료일', '다음특보시작일'])
ev = ev_all[ev_all.깨끗함 & (ev_all.분류 == '단일') & (ev_all.기간 == EVENT_PERIOD)].reset_index(drop=True)
print(f'{TEST_YEAR}년 깨끗한 단일 이벤트', len(ev), '개:', ev.groupby('종류').size().to_dict(), '| long_event(14일 초과):', int(ev.long_event.sum()), '개')
ev[['이벤트번호', '종류', '시작일', '종료일', '지속일수', '다음특보시작일', '다음특보종류']]

2025년 깨끗한 단일 이벤트 13 개: {'대설': 4, '폭염': 1, '한파': 3, '호우': 5} | long_event(14일 초과): 0 개


,이벤트번호,종류,시작일,종료일,지속일수,다음특보시작일,다음특보종류
0,단일-대설-25,대설,2025-01-05,2025-01-05,1,2025-01-08,한파
1,단일-한파-44,한파,2025-01-08,2025-01-11,4,NaT,NaN
2,단일-대설-27,대설,2025-03-04,2025-03-04,1,NaT,NaN
3,단일-대설-28,대설,2025-03-17,2025-03-18,2,NaT,NaN
4,단일-호우-71,호우,2025-05-16,2025-05-16,1,NaT,NaN
5,단일-호우-75,호우,2025-09-01,2025-09-01,1,2025-09-04,호우
6,단일-호우-76,호우,2025-09-04,2025-09-04,1,2025-09-05,폭염
7,단일-폭염-43,폭염,2025-09-05,2025-09-06,2,2025-09-13,호우
8,단일-호우-77,호우,2025-09-13,2025-09-13,1,2025-09-17,호우
9,단일-호우-78,호우,2025-09-17,2025-09-17,1,NaT,NaN


**9개 역 병렬 실행**: 역마다 한 번 적합하고 이벤트를 예측합니다.

In [28]:
jobs = [{'id': r.이벤트번호, 'start': r.시작일, 'end': r.종료일, 'H': int(r.지속일수) + EVENT_POST_DAYS} for r in ev.itertuples()]
out_ev = Parallel(n_jobs=WORKERS)(delayed(event_station_task)(st, jobs) for st in STATIONS)
paths = pd.concat([o[0] for o in out_ev], ignore_index=True)
ev_fits = pd.DataFrame([o[1] for o in out_ev])
print('일별 예측 표', paths.shape, '| 미수렴:', int((~ev_fits.수렴).sum()), '/', len(ev_fits), '| 뺀 더미:', sorted(set(','.join(ev_fits['뺀 더미']).split(',')) - {''}))

일별 예측 표 (1818, 11) | 미수렴: 0 / 9 | 뺀 더미: ['fare_2025']


**이벤트 × 역 결과 만들기**: 고정한 k(`SHOCK_K`)와 m(`RECOVERY_M`)로 모든 이벤트 × 9개 역의 충격·회복을 계산하고, 02번의 이벤트 정보(종류, 기간, 다음 특보)를 붙입니다. `results/test2025_event_effects.csv`로 저장합니다.

In [29]:
st_type = d_all.drop_duplicates('station').set_index('station').group
meta = ev.set_index('이벤트번호')[['분류', '종류', '시작일', '종료일', '지속일수', '기간', 'long_event', '다음특보시작일', '다음특보종류']]
st_i = {s: i for i, s in enumerate(STATIONS)}
rows = []
for (eid, st), g in paths.groupby(['id', '역']):
    m_ = meta.loc[eid]
    r = eval_event(g.set_index('date'), m_.시작일, m_.종료일, SHOCK_K, RECOVERY_M)
    r.pop('지속일수')
    rows.append({'이벤트번호': eid, '역': st, '유형': st_type[st], **r})
eff = pd.DataFrame(rows).merge(meta.reset_index(), on='이벤트번호')
eff['_s'] = eff.역.map(st_i)
eff = eff.sort_values(['시작일', '이벤트번호', '_s']).drop(columns='_s').reset_index(drop=True)
eff['참고용'] = eff.종류.eq('대설') | (eff.groupby('종류').이벤트번호.transform('nunique') < MIN_EVENTS)      # 이벤트가 적어 전부 참고용
cols = ['이벤트번호', '분류', '종류', '시작일', '종료일', '지속일수', '기간', 'long_event', '참고용', '역', '유형', 'n_below', '충격', '최대감소율%', '평균편차%', '기준선평균', '실제평균',
        '누적손실_기간', '누적손실_회복까지', '확인창일수', '확인창제한', '막힌날(해제후일차)', '회복일', '회복지연일', '중도절단', '다음특보시작일', '다음특보종류']
eff = eff[cols]
eff.to_csv(RESULTS / 'test2025_event_effects.csv', index=False, encoding='utf-8-sig')
paths.to_csv(RESULTS / 'test2025_event_daily_paths.csv', index=False, encoding='utf-8-sig')
print('이벤트 × 역', eff.shape, '| 충격 건수', int(eff.충격.sum()), '/', len(eff))

이벤트 × 역 (117, 27) | 충격 건수 3 / 117


**이벤트 × 역 표 (같은 이벤트에 대한 9개 역의 반응을 나란히)**: 최대 감소율(%)과 회복 지연(일)을 이벤트 줄 × 역 칸으로 펼칩니다. 회복 지연은 충격이 있었던 칸만 값이 있고(중도절단은 `>N`, N은 확인한 일수), 충격이 없으면 비웁니다. 아래 표는 **서술용**이며 이벤트가 적어 일반화하지 않습니다.

In [30]:
idx = ['이벤트번호', '종류', '시작일', '지속일수']
wide_drop = eff.pivot_table(index=idx, columns='역', values='최대감소율%', observed=True)[STATIONS].round(1)
eff['회복표시'] = eff.apply(lambda r: '' if not r.충격 else (f'>{int(r.확인창일수)}' if r.중도절단 else str(int(r.회복지연일))), axis=1)
wide_rec = eff.pivot(index=idx, columns='역', values='회복표시')[STATIONS]
wide_drop.to_csv(RESULTS / 'test2025_event_by_station_maxdrop.csv', encoding='utf-8-sig')
wide_rec.to_csv(RESULTS / 'test2025_event_by_station_recovery.csv', encoding='utf-8-sig')
wide_drop

,,,역,시청,역삼,선릉,신촌,혜화,신사,서울대입구,수유,연신내
이벤트번호,종류,시작일,지속일수,,,,,,,,,
단일-대설-25,대설,2025-01-05,1,23.9,-1.3,-4.6,-6.4,4.9,2.7,6.4,3.3,4.6
단일-대설-27,대설,2025-03-04,1,5.9,1.6,3.1,-2.8,12.0,5.3,-0.8,0.8,2.5
단일-대설-28,대설,2025-03-17,2,-1.1,6.0,4.5,-3.3,-0.9,0.3,1.8,2.0,1.3
단일-대설-29,대설,2025-12-04,1,-1.2,-8.1,-4.4,-4.6,-9.5,-8.3,-0.6,-1.8,-0.6
단일-폭염-43,폭염,2025-09-05,2,10.3,-0.3,-1.1,2.4,10.0,-2.1,3.3,3.5,5.1
단일-한파-44,한파,2025-01-08,4,7.8,2.2,2.5,8.9,11.7,5.0,6.7,6.7,7.5
단일-한파-45,한파,2025-11-02,2,-1.0,2.4,-0.1,2.7,-1.2,1.1,1.6,1.6,2.4
단일-한파-46,한파,2025-12-02,2,1.7,-1.5,0.9,2.8,2.9,2.5,4.3,4.0,5.3
단일-호우-71,호우,2025-05-16,1,0.9,-1.1,0.0,2.8,-0.1,2.5,1.9,4.5,2.9


**충격·회복 표시 (이벤트 × 역)**: 충격이 있었던 칸(`*`)과 그 회복 지연을 한눈에 봅니다 (빈칸 = 충격 없음, `*2` = 충격 있음·회복 지연 2일, `*>5` = 충격 있음·5일 확인했으나 중도절단).

In [31]:
mark = eff.pivot(index=idx, columns='역', values='충격')[STATIONS]
disp = mark.copy().astype(object)
for ix in mark.index:
    for s in STATIONS:
        disp.loc[ix, s] = ('*' + wide_rec.loc[ix, s]) if mark.loc[ix, s] else ''
disp

,,,역,시청,역삼,선릉,신촌,혜화,신사,서울대입구,수유,연신내
이벤트번호,종류,시작일,지속일수,,,,,,,,,
단일-대설-25,대설,2025-01-05,1,*1,,,,,,,,
단일-대설-27,대설,2025-03-04,1,,,,,,,,,
단일-대설-28,대설,2025-03-17,2,,,,,,,,,
단일-대설-29,대설,2025-12-04,1,,,,,,,,,
단일-폭염-43,폭염,2025-09-05,2,,,,,,,,,
단일-한파-44,한파,2025-01-08,4,,,,,,,,*1,*1
단일-한파-45,한파,2025-11-02,2,,,,,,,,,
단일-한파-46,한파,2025-12-02,2,,,,,,,,,
단일-호우-71,호우,2025-05-16,1,,,,,,,,,


**결과 요약 함수 `summarize`**: 04번과 같은 코드입니다 (유형 × 종류별 이벤트 수, 충격 비율, 최대 감소율, 회복 판정 가능 건수·이벤트 수(중도절단 제외), 회복 지연, 중도절단 비율). 이벤트가 적어 전부 참고용으로 표시합니다.

In [32]:
def summarize(df):
    rows = []
    for (typ, kind), g in df.groupby(['유형', '종류']):
        sh = g[g.충격]
        rec = sh[~sh.중도절단]
        rows.append({'유형': typ, '종류': kind, '참고용': '참고용' if g.참고용.any() else '', '이벤트 수': g.이벤트번호.nunique(), '이벤트×역': len(g),
                     '충격 비율': round(g.충격.mean(), 3), '최대감소율(평균%)': round(g['최대감소율%'].mean(), 1), '최대감소율(중앙%)': round(g['최대감소율%'].median(), 1),
                     '충격 건수': len(sh), '회복 판정 가능 건수(이벤트×역, 중도절단 제외)': len(rec), '회복 판정 가능 이벤트 수(중도절단 제외)': rec.이벤트번호.nunique(),
                     '회복지연(평균일)': round(rec.회복지연일.mean(), 1) if len(rec) else np.nan, '회복지연(중앙일)': rec.회복지연일.median() if len(rec) else np.nan,
                     '중도절단 비율(충격 중)': round(sh.중도절단.mean(), 3) if len(sh) else np.nan, '누적손실(평균, 명)': round(g.누적손실_회복까지.mean())})
    t = pd.DataFrame(rows)
    t['유형'] = pd.Categorical(t.유형, TYPE_ORDER, ordered=True)
    return t.sort_values(['종류', '유형']).reset_index(drop=True)

**2025년 요약 (단일 이벤트, 유형 × 종류)**: `results/test2025_event_summary.csv`로 저장합니다.

In [33]:
t_test = summarize(eff)
t_test.to_csv(RESULTS / 'test2025_event_summary.csv', index=False, encoding='utf-8-sig')
t_test

,유형,종류,참고용,이벤트 수,이벤트×역,충격 비율,최대감소율(평균%),최대감소율(중앙%),충격 건수,"회복 판정 가능 건수(이벤트×역, 중도절단 제외)",회복 판정 가능 이벤트 수(중도절단 제외),회복지연(평균일),회복지연(중앙일),중도절단 비율(충격 중),"누적손실(평균, 명)"
0,Employment,대설,참고용,4,12,0.083,2.0,0.3,1,1,1,1.0,1.0,0.0,1653
1,Commercial & Leisure,대설,참고용,4,12,0.000,-0.9,-1.8,0,0,0,NaN,NaN,NaN,-1240
2,Residential,대설,참고용,4,12,0.000,1.6,1.6,0,0,0,NaN,NaN,NaN,890
3,Employment,폭염,참고용,1,3,0.000,3.0,-0.3,0,0,0,NaN,NaN,NaN,-7430
4,Commercial & Leisure,폭염,참고용,1,3,0.000,3.5,2.4,0,0,0,NaN,NaN,NaN,-43
5,Residential,폭염,참고용,1,3,0.000,3.9,3.5,0,0,0,NaN,NaN,NaN,1678
6,Employment,한파,참고용,3,9,0.000,1.6,1.7,0,0,0,NaN,NaN,NaN,-1239
7,Commercial & Leisure,한파,참고용,3,9,0.000,4.1,2.8,0,0,0,NaN,NaN,NaN,5437
8,Residential,한파,참고용,3,9,0.222,4.5,4.3,2,2,1,1.0,1.0,0.0,4803
9,Employment,호우,참고용,5,15,0.000,-0.7,0.1,0,0,0,NaN,NaN,NaN,-1843


**2014~2024 결과와 방향 비교**: 04번의 단일(주 분석) 결과(2014~2024)와 2025 결과를 종류별·유형별로 나란히 놓습니다. 2025는 이벤트가 종류당 1~5개뿐이라 **우연히 갈릴 수 있어 방향이 같은지만 서술**합니다 (충격 비율과 최대 감소율의 크기·순서).

In [34]:
eff_hist = pd.read_csv(RESULTS / 'event_effects.csv')
eff_hist = eff_hist[eff_hist.분류 == '단일']
def kind_stats(df):
    return pd.DataFrame({'이벤트 수': df.groupby('종류').이벤트번호.nunique(), '충격 비율': df.groupby('종류').충격.mean().round(3), '최대감소율(평균%)': df.groupby('종류')['최대감소율%'].mean().round(1)})
ks = pd.concat({'2014~2024': kind_stats(eff_hist), str(TEST_YEAR): kind_stats(eff)}, axis=1).reindex(list(EVENT_TYPES))
ks[('방향', '충격 비율 2025 ≥ 2014~2024')] = ks[(str(TEST_YEAR), '충격 비율')] >= ks[('2014~2024', '충격 비율')]
ks

2014~2024                    2025                                       방향
       이벤트 수  충격 비율 최대감소율(평균%) 이벤트 수  충격 비율 최대감소율(평균%) 충격 비율 2025 ≥ 2014~2024
종류                                                                           
폭염        27  0.119        4.8     1  0.000        3.5                  False
한파        19  0.140        4.2     3  0.074        3.4                  False
호우        39  0.285        7.0     5  0.000        0.6                  False
대설         8  0.014        0.3     4  0.028        0.9                   True

**유형 × 종류별 비교**: 같은 비교를 역 유형까지 나눠 봅니다 (충격 비율, 최대 감소율 평균). 칸마다 2025 이벤트가 1~5개라 크기는 참고만 하세요.

In [35]:
def cell_stats(df):
    g = df.groupby(['유형', '종류'])
    return pd.DataFrame({'충격 비율': g.충격.mean().round(3), '최대감소율(평균%)': g['최대감소율%'].mean().round(1)})
cmp_ev = pd.concat({'2014~2024': cell_stats(eff_hist), str(TEST_YEAR): cell_stats(eff)}, axis=1)
cmp_ev.index = pd.MultiIndex.from_tuples(cmp_ev.index, names=['유형', '종류'])
cmp_ev = cmp_ev.reindex(pd.MultiIndex.from_product([TYPE_ORDER, list(EVENT_TYPES)], names=['유형', '종류']))
flat = cmp_ev.copy()
flat.columns = [f'{p}_{q}' for p, q in flat.columns]                  # 열 이름을 '기간_지표'로 평평하게
flat.reset_index().to_csv(RESULTS / 'test2025_event_vs_2014_2024.csv', index=False, encoding='utf-8-sig')
cmp_ev

2014~2024              2025           
                            충격 비율 최대감소율(평균%)  충격 비율 최대감소율(평균%)
유형                   종류                                       
Employment           폭염     0.148        5.6  0.000        3.0
                     한파     0.053        3.2  0.000        1.6
                     호우     0.128        4.8  0.000       -0.7
                     대설     0.042        0.1  0.083        2.0
Commercial & Leisure 폭염     0.074        5.0  0.000        3.5
                     한파     0.123        4.3  0.000        4.1
                     호우     0.222        8.1  0.000        1.3
                     대설     0.000       -0.6  0.000       -0.9
Residential          폭염     0.136        3.9  0.000        3.9
                     한파     0.246        5.1  0.222        4.5
                     호우     0.504        8.3  0.000        1.2
                     대설     0.000        1.2  0.000        1.6

## 실행 기록

2025년 test를 **이번 한 번만** 실행했습니다. 실행 일시와 고정한 설정을 `results/test2025_run_info.json`에 기록합니다 (이 파일이 있으면 노트북을 다시 실행할 수 없음).

In [36]:
info = {'run_datetime': datetime.now().astimezone().isoformat(timespec='seconds'), 'test_year': TEST_YEAR, 'run_count': 1,
        'retrain': f'2014-01-01 ~ {RETRAIN_END}', 'evaluation': f'{TEST_START.date()} ~ {TEST_END.date()}',
        'fixed_settings': {'ORDER': list(ORDER), 'SEASONAL_ORDER': list(SEASONAL_ORDER), 'MASK_COLS': MASK_COLS, 'STEP_COLS': STEP_COLS, 'SEASON_COLS': SEASON_COLS, 'STATION_STEP': STATION_STEP,
                           'ALERT_MASK_AFTER_DAYS': ALERT_MASK_AFTER_DAYS, 'SHOCK_K': SHOCK_K, 'RECOVERY_M': RECOVERY_M, 'FIT_METHOD': FIT_METHOD, 'FIT_MAXITER': FIT_MAXITER},
        'models': MODELS, 'versions': {'statsmodels': statsmodels.__version__, 'pandas': pd.__version__, 'numpy': np.__version__},
        'note': '2025년 test는 이번 한 번만 실행했고, 2025 결과를 보고 설정을 바꾸지 않았다.'}
RUN_INFO.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(info, ensure_ascii=False, indent=2))

{
  "run_datetime": "2026-11-13T16:34:19+09:00",
  "test_year": 2025,
  "run_count": 1,
  "retrain": "2014-01-01 ~ 2024-12-31",
  "evaluation": "2025-01-01 ~ 2025-12-31",
  "fixed_settings": {
    "ORDER": [
      1,
      0,
      1
    ],
    "SEASONAL_ORDER": [
      1,
      1,
      1,
      7
    ],
    "MASK_COLS": [
      "holiday",
      "labor_day",
      "myeongjeol_pm2",
      "covid",
      "mers",
      "strike",
      "rally_cityhall",
      "alert_unknown"
    ],
    "STEP_COLS": [
      "fare_2015",
      "fare_2023",
      "fare_2025",
      "climate_card"
    ],
    "SEASON_COLS": [
      "summer_vacation"
    ],
    "STATION_STEP": {
      "서울대입구": "sillim_line",
      "신사": "sinbundang_ext"
    },
    "ALERT_MASK_AFTER_DAYS": 7,
    "SHOCK_K": 1,
    "RECOVERY_M": 2,
    "FIT_METHOD": "lbfgs",
    "FIT_MAXITER": 1000
  },
  "models": [
    "Naive",
    "N",
    "A",
    "B_lag",
    "B",
    "C"
  ],
  "versions": {
    "statsmodels": "0.14.6",
    "pandas": "2.3.3